# Physics-Informed Graph Neural Network for River Modeling (Delft3D FM)

This notebook trains a PIGNN to model river hydrodynamics based on the 2D Shallow Water Equations, utilizing a mesh generated by Delft3D FM. It forces the network strictly using real physical boundaries.

In [ ]:
# 1. Download Data directly from GitHub
!git clone https://github.com/ATIK2110018/gironde_PINN.git
%cd gironde_PINN

In [ ]:
# 2. Install required libraries
!pip install torch-geometric netCDF4 xarray matplotlib scipy

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
import netCDF4 as nc
import numpy as np
import math
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.collections as mcoll

# Setup GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ==========================================
# 1. DATA EXTRACTION FUNCTIONS
# ==========================================
def extract_graph_from_netcdf(nc_file_path):
    print(f"Reading mesh from {nc_file_path}...")
    dataset = nc.Dataset(nc_file_path, 'r')
    
    if 'mesh2d_node_x' in dataset.variables:
        node_x = dataset.variables['mesh2d_node_x'][:]
        node_y = dataset.variables['mesh2d_node_y'][:]
        edge_nodes = dataset.variables['mesh2d_edge_nodes'][:]
        node_z = dataset.variables['mesh2d_node_z'][:] if 'mesh2d_node_z' in dataset.variables else np.zeros_like(node_x)
    elif 'NetNode_x' in dataset.variables:
        node_x = dataset.variables['NetNode_x'][:]
        node_y = dataset.variables['NetNode_y'][:]
        edge_nodes = dataset.variables['NetLink'][:]
        node_z = dataset.variables['NetNode_z'][:] if 'NetNode_z' in dataset.variables else np.zeros_like(node_x)
    else:
        raise ValueError("Could not find standard node coordinate variables.")

    is_spherical = (np.max(node_x) < 180.0) and (np.min(node_x) > -180.0)
    if is_spherical:
        print("Spherical coordinates (Lat/Lon) detected. Edge lengths will be converted to meters for PDE physics computation.")
        mean_lat = np.mean(node_y)
        lat_to_m = 111139.0
        lon_to_m = 111139.0 * math.cos(math.radians(mean_lat))
    else:
        lat_to_m = 1.0
        lon_to_m = 1.0

    edge_index_list, edge_attr_list = [], []
    for i in range(edge_nodes.shape[1] if edge_nodes.shape[0] == 2 else edge_nodes.shape[0]):
        if edge_nodes.shape[0] == 2:
            n1 = int(edge_nodes[0, i]) - 1
            n2 = int(edge_nodes[1, i]) - 1
        else:
            n1 = int(edge_nodes[i, 0]) - 1
            n2 = int(edge_nodes[i, 1]) - 1
            
        if n1 >= 0 and n2 >= 0:
            edge_index_list.extend([[n1, n2], [n2, n1]])
            dx = node_x[n2] - node_x[n1]
            dy = node_y[n2] - node_y[n1]
            
            dx_m = dx * lon_to_m
            dy_m = dy * lat_to_m
            dist_m = math.sqrt(dx_m**2 + dy_m**2)
            
            edge_attr_list.extend([[dx_m, dy_m, dist_m], [-dx_m, -dy_m, dist_m]])
            
    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr_list, dtype=torch.float32)
    node_coords = torch.tensor(np.column_stack((node_x, node_y)), dtype=torch.float32)
    node_z = torch.tensor(node_z, dtype=torch.float32)
    dataset.close()
    return node_coords, edge_index, edge_attr, node_z

def load_friction_xyz(filepath, node_coords):
    print(f"Loading friction from {filepath}...")
    data = np.loadtxt(filepath)
    val_interp = griddata((data[:, 0], data[:, 1]), data[:, 2], (node_coords[:, 0], node_coords[:, 1]), method='nearest')
    return torch.tensor(val_interp, dtype=torch.float32)

def load_boundary_pli(filepath, node_coords, threshold=0.002):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    coords = np.array([[float(p[0]), float(p[1])] for line in lines[2:] if len(p := line.strip().split()) >= 2])
    node_coords_np = node_coords.numpy()
    boundary_nodes = []
    for i in range(len(coords)-1):
        p1, p2 = coords[i], coords[i+1]
        l2 = np.sum((p2 - p1)**2)
        if l2 == 0: continue
        t = np.clip(np.sum((node_coords_np - p1) * (p2 - p1), axis=1) / l2, 0, 1)
        projection = p1 + t[:, np.newaxis] * (p2 - p1)
        dist = np.sqrt(np.sum((node_coords_np - projection)**2, axis=1))
        boundary_nodes.extend(np.where(dist < threshold)[0])
    return list(set(boundary_nodes))

def load_boundary_bc(filepath):
    times, values = [], []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                try:
                    times.append(float(parts[0]))
                    values.append(float(parts[1]))
                except ValueError:
                    continue
    return np.array(times), np.array(values)

def load_truth_data(nc_file_path, node_coords):
    print(f"Loading true state data from {nc_file_path} for data loss...")
    dataset = nc.Dataset(nc_file_path, 'r')
    times = dataset.variables['time'][:]
    
    eta_raw = dataset.variables['mesh2d_s1'][:]
    u_raw = dataset.variables['mesh2d_ucx'][:]
    v_raw = dataset.variables['mesh2d_ucy'][:]
    
    # Handle missing/masked values by filling with 0
    eta_raw = np.ma.filled(eta_raw, fill_value=0.0)
    u_raw = np.ma.filled(u_raw, fill_value=0.0)
    v_raw = np.ma.filled(v_raw, fill_value=0.0)
    
    if eta_raw.shape[1] == node_coords.shape[0]:
        print("Truth data maps 1:1 with nodes.")
        eta_nodes, u_nodes, v_nodes = eta_raw, u_raw, v_raw
    else:
        print("Truth data is at faces. Fast interpolating to nodes using KDTree...")
        face_x = dataset.variables['mesh2d_face_x'][:]
        face_y = dataset.variables['mesh2d_face_y'][:]
        face_coords = np.column_stack((face_x, face_y))
        
        tree = cKDTree(face_coords)
        node_coords_np = node_coords.cpu().numpy()
        _, indices = tree.query(node_coords_np)
        
        eta_nodes = eta_raw[:, indices]
        u_nodes = u_raw[:, indices]
        v_nodes = v_raw[:, indices]
        
    dataset.close()
    return times, torch.tensor(eta_nodes, dtype=torch.float32), torch.tensor(u_nodes, dtype=torch.float32), torch.tensor(v_nodes, dtype=torch.float32)

In [ ]:
# ==========================================
# 2. GRAPH NEURAL NETWORK ARCHITECTURE
# ==========================================
class HydroMPNNLayer(MessagePassing):
    def __init__(self, node_in_dim, edge_in_dim, out_dim):
        super(HydroMPNNLayer, self).__init__(aggr='mean')
        self.message_mlp = nn.Sequential(nn.Linear(node_in_dim * 2 + edge_in_dim, 64), nn.ReLU(), nn.Linear(64, 64))
        self.update_mlp = nn.Sequential(nn.Linear(node_in_dim + 64, 64), nn.ReLU(), nn.Linear(64, out_dim))
    def forward(self, x, edge_index, edge_attr): return self.propagate(edge_index, x=x, edge_attr=edge_attr)
    def message(self, x_i, x_j, edge_attr): return self.message_mlp(torch.cat([x_i, x_j, edge_attr], dim=1))
    def update(self, aggr_out, x): return self.update_mlp(torch.cat([x, aggr_out], dim=1))

class RiverPIGNN(nn.Module):
    def __init__(self, node_features_dim=5, hidden_dim=64, edge_dim=3):
        super(RiverPIGNN, self).__init__()
        self.encoder = nn.Linear(node_features_dim, hidden_dim)
        self.mpnn1 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.mpnn2 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.decoder = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.encoder(x))
        h = F.relu(self.mpnn1(h, edge_index, edge_attr))
        h = self.mpnn2(h, edge_index, edge_attr)
        delta = self.decoder(h)
        
        delta_eta = torch.clamp(delta[:, 0], min=-0.5, max=0.5)
        delta_u = torch.clamp(delta[:, 1], min=-0.5, max=0.5)
        delta_v = torch.clamp(delta[:, 2], min=-0.5, max=0.5)
        
        out_eta = x[:, 0] + delta_eta
        out_u = x[:, 1] + delta_u
        out_v = x[:, 2] + delta_v
        
        zb = x[:, 3]
        out_eta = torch.clamp(out_eta, min=zb + 0.05, max=zb + 25.0) 
        out_u = torch.clamp(out_u, min=-10.0, max=10.0)             
        out_v = torch.clamp(out_v, min=-10.0, max=10.0)
        
        return torch.stack([out_eta, out_u, out_v], dim=1)

In [ ]:
# ==========================================
# 3. PHYSICS LOSS 
# ==========================================
def physics_loss_swe(state_t, state_t_next, edge_index, edge_attr, dt=60.0, g=9.81):
    eta_t, u_t, v_t, zb, cf = state_t.T
    h_t = torch.clamp(eta_t - zb, min=0.05) 
    
    eta_next, u_next, v_next = state_t_next.T
    deta_dt, du_dt, dv_dt = (eta_next - eta_t)/dt, (u_next - u_t)/dt, (v_next - v_t)/dt
    
    row, col = edge_index
    dx, dy, dist = edge_attr.T
    dist_clamped = torch.clamp(dist, min=1.0)
    
    delta_eta = eta_t[col] - eta_t[row]
    delta_u = u_t[col] - u_t[row]
    delta_v = v_t[col] - v_t[row]
    delta_hu = (h_t[col] * u_t[col]) - (h_t[row] * u_t[row])
    delta_hv = (h_t[col] * v_t[col]) - (h_t[row] * v_t[row])
    
    weight_x, weight_y = dx / (dist_clamped ** 2 + 1e-8), dy / (dist_clamped ** 2 + 1e-8)
    num_nodes = eta_t.size(0)
    
    def scatter_add_pt(src, index, dim_size):
        return torch.zeros(dim_size, dtype=src.dtype, device=src.device).scatter_add_(0, index, src)
    
    grad_eta_x, grad_eta_y = scatter_add_pt(delta_eta * weight_x, row, num_nodes), scatter_add_pt(delta_eta * weight_y, row, num_nodes)
    grad_u_x, grad_u_y = scatter_add_pt(delta_u * weight_x, row, num_nodes), scatter_add_pt(delta_u * weight_y, row, num_nodes)
    grad_v_x, grad_v_y = scatter_add_pt(delta_v * weight_x, row, num_nodes), scatter_add_pt(delta_v * weight_y, row, num_nodes)
    grad_hu_x, grad_hv_y = scatter_add_pt(delta_hu * weight_x, row, num_nodes), scatter_add_pt(delta_hv * weight_y, row, num_nodes)
    
    mass_residual = deta_dt + grad_hu_x + grad_hv_y
    velocity_magnitude = torch.sqrt(u_t**2 + v_t**2 + 1e-8)
    momentum_x_residual = du_dt + u_t * grad_u_x + v_t * grad_u_y + g * grad_eta_x + cf * u_t * velocity_magnitude / h_t
    momentum_y_residual = dv_dt + u_t * grad_v_x + v_t * grad_v_y + g * grad_eta_y + cf * v_t * velocity_magnitude / h_t
    
    return torch.mean(mass_residual**2) + torch.mean(momentum_x_residual**2) + torch.mean(momentum_y_residual**2)

### Parse Real Data and Boundaries

In [ ]:
netcdf_path = "data/input/FlowFM_net.nc"
node_coords, edge_index, edge_attr, node_z = extract_graph_from_netcdf(netcdf_path)
num_nodes = node_coords.size(0)

friction = load_friction_xyz("data/input/frictioncoefficient.xyz", node_coords)

bnd_port = load_boundary_pli("data/input/port block.pli", node_coords, threshold=0.002)
bnd_garonne = load_boundary_pli("data/input/garonne.pli", node_coords, threshold=0.002)
bnd_dordogne = load_boundary_pli("data/input/dordogne.pli", node_coords, threshold=0.002)

t_port, v_port = load_boundary_bc("data/input/WaterLevel.bc")
t_garonne, v_garonne = load_boundary_bc("data/input/garonne.bc")
t_dordogne, v_dordogne = load_boundary_bc("data/input/dordogne.bc")

print(f"Successfully parsed fully realistic setup!")
print(f"Port Boundary Nodes: {len(bnd_port)}")
print(f"Garonne Boundary Nodes: {len(bnd_garonne)}")
print(f"Dordogne Boundary Nodes: {len(bnd_dordogne)}")

### Pre-Training Visualizations

In [ ]:
x = node_coords[:, 0].numpy()
y = node_coords[:, 1].numpy()
z = node_z.numpy()

plt.figure(figsize=(15, 4))
plt.subplot(1, 2, 1)
plt.plot(t_port/3600, v_port, 'r-')
plt.title("Port Boundary (Water Level)")
plt.xlabel("Time (hours)")
plt.ylabel("Water Level (m)")

plt.subplot(1, 2, 2)
plt.plot(t_garonne/3600, v_garonne, 'g-', label="Garonne")
plt.plot(t_dordogne/3600, v_dordogne, 'orange', label="Dordogne")
plt.title("River Boundaries (Discharge)")
plt.xlabel("Time (hours)")
plt.ylabel("Discharge (m3/s)")
plt.legend()
plt.tight_layout()
plt.show()

edges_np = edge_index.numpy()
lines = [[(x[edges_np[0, i]], y[edges_np[0, i]]), (x[edges_np[1, i]], y[edges_np[1, i]])] 
         for i in range(0, edges_np.shape[1], 2)]
lc = mcoll.LineCollection(lines, colors='k', linewidths=0.1, alpha=0.3)

plt.figure(figsize=(10, 8))
ax1 = plt.gca()
ax1.add_collection(lc)
ax1.autoscale()
plt.scatter(x[bnd_port], y[bnd_port], c='red', s=50, zorder=5, label='Port (Water Level)')
plt.scatter(x[bnd_garonne], y[bnd_garonne], c='lime', s=50, zorder=5, label='Garonne (Discharge)')
plt.scatter(x[bnd_dordogne], y[bnd_dordogne], c='magenta', s=50, zorder=5, label='Dordogne (Discharge)')
plt.title("Exact Neural Network Mesh showing ALL boundary nodes along the polylines")
plt.legend()
plt.axis('equal')
plt.show()

### Training Loop (Multi-Epoch)

In [ ]:
node_coords = node_coords.to(device)
edge_index = edge_index.to(device)
edge_attr = edge_attr.to(device)
node_z = node_z.to(device)
friction = friction.to(device)

model = RiverPIGNN(node_features_dim=5, hidden_dim=64, edge_dim=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

dt = 60.0  
max_time = t_port[-1] if len(t_port) > 0 else 36000
num_epochs = 100 

def get_interp_val(t, times, values):
    return np.interp(t, times, values)

print(f"Starting Dynamic Physics Solver (dt={dt}s) for {num_epochs} Epochs...")
for epoch in range(num_epochs):
    print(f"\n--- EPOCH {epoch+1}/{num_epochs} ---")
    current_time = 0.0
    
    # Start water level at Sea Level (0.0m) instead of the top of the highest mountain!
    # If the riverbed is above sea level (dry land), just put a 10cm puddle so it doesn't divide by zero.
    initial_water_level = torch.clamp(torch.tensor(0.0, device=device), min=node_z + 0.1)
    state_t = torch.zeros((num_nodes, 5), dtype=torch.float32).to(device)
    state_t[:, 0] = initial_water_level
    state_t[:, 3] = node_z
    state_t[:, 4] = friction
    
    # Ultimate brute-force file search to guarantee we find the map.nc file in Kaggle
    map_files = []
    for search_dir in ['/kaggle', '.']:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                for f in files:
                    if 'map.nc' in f.lower() or 'flowfm_map.nc' in f.lower():
                        map_files.append(os.path.join(root, f))
    
    if map_files:
        map_file_path = map_files[0]
        print(f"SUCCESS: Found data file at {map_file_path}")
        true_times, true_eta, true_u, true_v = load_truth_data(map_file_path, node_coords)
        has_truth_data = True
    else:
        print(f"WARNING: FlowFM_map.nc not found anywhere! Data loss will be 0.0.")
        has_truth_data = False
    
    while current_time < max_time:
        model.train()
        optimizer.zero_grad()
        
        predicted_state_next = model(state_t, edge_index, edge_attr)
        loss_physics = physics_loss_swe(state_t, predicted_state_next, edge_index, edge_attr, dt=dt)
        
        target_port_wl = get_interp_val(current_time + dt, t_port, v_port)
        target_gar_q = get_interp_val(current_time + dt, t_garonne, v_garonne) * 0.001
        target_dor_q = get_interp_val(current_time + dt, t_dordogne, v_dordogne) * 0.001
        
        loss_bnd_port = F.mse_loss(predicted_state_next[bnd_port, 0], torch.tensor(target_port_wl, dtype=torch.float32, device=device).expand(len(bnd_port)))
        loss_bnd_gar = F.mse_loss(predicted_state_next[bnd_garonne, 1], torch.tensor(target_gar_q, dtype=torch.float32, device=device).expand(len(bnd_garonne)))
        loss_bnd_dor = F.mse_loss(predicted_state_next[bnd_dordogne, 1], torch.tensor(target_dor_q, dtype=torch.float32, device=device).expand(len(bnd_dordogne)))
        
        loss_boundary = loss_bnd_port + loss_bnd_gar + loss_bnd_dor
        
        # Supervised Data Loss implementation with continuous time interpolation
        if has_truth_data:
            t_target = current_time + dt
            
            if t_target <= true_times[0]:
                target_eta = true_eta[0].to(device)
                target_u = true_u[0].to(device)
                target_v = true_v[0].to(device)
            elif t_target >= true_times[-1]:
                target_eta = true_eta[-1].to(device)
                target_u = true_u[-1].to(device)
                target_v = true_v[-1].to(device)
            else:
                idx2 = np.searchsorted(true_times, t_target)
                idx1 = idx2 - 1
                t1, t2 = true_times[idx1], true_times[idx2]
                w = (t_target - t1) / (t2 - t1 + 1e-8)
                
                target_eta = ((1 - w) * true_eta[idx1] + w * true_eta[idx2]).to(device)
                target_u = ((1 - w) * true_u[idx1] + w * true_u[idx2]).to(device)
                target_v = ((1 - w) * true_v[idx1] + w * true_v[idx2]).to(device)
            
            loss_data_eta = F.mse_loss(predicted_state_next[:, 0], target_eta)
            loss_data_u = F.mse_loss(predicted_state_next[:, 1], target_u)
            loss_data_v = F.mse_loss(predicted_state_next[:, 2], target_v)
            loss_data = loss_data_eta + loss_data_u + loss_data_v
        else:
            loss_data = torch.tensor(0.0, device=device)
        
        # Lower physics weight temporarily so the Boundary condition has the strength to move the water!
        total_loss = (0.001 * loss_physics) + (100.0 * loss_boundary) + (1.0 * loss_data)
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        if current_time % 3600 == 0: 
            print(f"Time: {current_time/3600:.1f}h | Total: {total_loss.item():.2f} | Phys: {loss_physics.item():.2f} | Bnd: {loss_boundary.item():.2f} | Data: {loss_data.item():.2f}")
            
        state_t = torch.zeros((num_nodes, 5), dtype=torch.float32, device=device)
        state_t[:, :3] = predicted_state_next.detach().clone()
        state_t[:, 3] = node_z
        state_t[:, 4] = friction
        current_time += dt

print("Training Complete!")
torch.save(model.state_dict(), "pignn_weights_sim.pth")

### Post-Simulation Visualization (Water Level & Velocity)

In [ ]:
x_np = node_coords[:, 0].cpu().numpy()
y_np = node_coords[:, 1].cpu().numpy()
triang = mtri.Triangulation(x_np, y_np)
triangles = triang.triangles
xtri = x_np[triangles] - np.roll(x_np[triangles], 1, axis=1)
ytri = y_np[triangles] - np.roll(y_np[triangles], 1, axis=1)
maxi_local = np.max(np.sqrt(xtri**2 + ytri**2), axis=1)
triang.set_mask(maxi_local > 0.02)

water_level = state_t[:, 0].cpu().numpy()
u = state_t[:, 1].cpu().numpy()
v = state_t[:, 2].cpu().numpy()
velocity_mag = np.sqrt(u**2 + v**2)

plt.figure(figsize=(18, 6))

plt.subplot(1, 2, 1)
plt.tripcolor(triang, water_level, cmap='Blues')
plt.colorbar(label='Water Surface Elevation (m)')
plt.title(f"Final Predicted Water Level at T={current_time/3600:.1f}h")
plt.axis('equal')

plt.subplot(1, 2, 2)
plt.tripcolor(triang, velocity_mag, cmap='viridis')
plt.colorbar(label='Velocity Magnitude (m/s)')
plt.title(f"Final Predicted Velocity at T={current_time/3600:.1f}h")
plt.axis('equal')

plt.show()